# Scenario 4 – Great Expectations Streaming Validation

This notebook demonstrates **data quality validation using Great Expectations** integrated with Spark.

Covered in this notebook:
- Expectation suite definition (inlined for Databricks compatibility)
- `foreachBatch`-style validation handler (same function used in streaming pipelines)
- Fail-fast behavior on validation errors
- Delta Lake output with validated data

**Note:** On Databricks interactive clusters, we run the validation in batch mode
to avoid DBFS/checkpoint restrictions. The `validate_with_gx` function is the same
handler used in production streaming via `foreachBatch`.

**Delta Live Tables (DLT) is NOT executed here** — DLT requires a dedicated pipeline
runtime. DLT implementation is in: `customer_dlt_streaming_pipeline/transformations/my_transformation.py`

In [ ]:
# -------------------------------------------------
# Install Great Expectations (pinned to v0.x API)
# The project uses ge.from_pandas() / ge_df.validate()
# which were removed in GX v1.0.
# -------------------------------------------------
%pip install "great_expectations>=0.15,<1.0" --quiet
dbutils.library.restartPython()

In [ ]:
# -------------------------------------------------
# Notebook parameters (passed from Airflow / Databricks Jobs)
# Re-read after restartPython()
# -------------------------------------------------
dbutils.widgets.text("environment", "local")
dbutils.widgets.text("run_date", "")
environment = dbutils.widgets.get("environment")

print(f"Starting GX streaming pipeline | env={environment}")

In [ ]:
# -------------------------------------------------
# GX Expectation Suite Definition
#
# Same rules as src/scenario_4_customer_validation_suite.py
# but self-contained so no local package import is needed.
# -------------------------------------------------
from datetime import date
import great_expectations as ge


def apply_customer_expectations(ge_df):
    """
    Apply Great Expectations validation rules to a customer DataFrame.

    Expectations (per assignment requirements):
    1. party_key: not null and unique within batch
    2. country: must be within allowed set {US, IN, UK, CA}
    3. source_updated_at: not null (timestamp type enforced by Spark schema)
    4. dob (if present): must be before today
    5. name: null percentage below 10% threshold (90%+ non-null)
    """
    # 1. party_key: non-null and unique
    ge_df.expect_column_values_to_not_be_null(column="party_key")
    ge_df.expect_column_values_to_be_unique(column="party_key")

    # 2. country: within allowed set
    ge_df.expect_column_values_to_be_in_set(
        column="country", value_set=["US", "IN", "UK", "CA"]
    )

    # 3. source_updated_at: not null
    #    Timestamp type validity is enforced upstream by Spark schema
    #    (TimestampType casting rejects non-timestamp values before GX runs)
    ge_df.expect_column_values_to_not_be_null(column="source_updated_at")

    # 4. dob: if present, must be <= today
    if "dob" in ge_df.columns:
        ge_df.expect_column_values_to_be_between(
            column="dob",
            min_value="1900-01-01",
            max_value=date.today().isoformat(),
            allow_cross_type_comparisons=True,
            mostly=1.0,
        )

    # 5. name: at least 90% non-null
    ge_df.expect_column_values_to_not_be_null(column="name", mostly=0.90)
    return ge_df


print("Expectation suite defined (5 requirements covered)")

In [ ]:
# -------------------------------------------------
# foreachBatch handler: validate + write each micro-batch
#
# ETL PIPELINE INTEGRATION:
# In production streaming, this function is passed to
# writeStream.foreachBatch(validate_with_gx).
# Each micro-batch is validated before being written to Delta.
#
# WHAT HAPPENS ON VALIDATION FAILURE:
# - RuntimeError is raised immediately (fail-fast)
# - The streaming query stops, no bad data is written
# - Airflow detects the task failure and triggers notify_failure
# - The batch can be retried after fixing upstream data
# -------------------------------------------------
from pyspark.sql import DataFrame

OUTPUT_TABLE = "gx_validated_customers"


def validate_with_gx(batch_df: DataFrame, batch_id: int):
    """
    Called for each micro-batch via foreachBatch (or directly in batch mode).

    Flow:
    1. Convert Spark DataFrame to Pandas (GX operates on Pandas)
    2. Apply full expectation suite
    3. FAIL FAST if any expectation fails (no bad data written)
    4. Write validated data to Delta table
    """
    if batch_df.isEmpty():
        print(f"Batch {batch_id}: empty, skipping")
        return

    pdf = batch_df.toPandas()
    ge_df = ge.from_pandas(pdf)
    ge_df = apply_customer_expectations(ge_df)
    results = ge_df.validate()

    if not results["success"]:
        failed = [
            r["expectation_config"]["expectation_type"]
            for r in results["results"]
            if not r["success"]
        ]
        raise RuntimeError(
            f"GX validation failed in batch {batch_id}. "
            f"Failed expectations: {failed}"
        )

    print(f"Batch {batch_id}: GX validation passed ({len(pdf)} rows)")

    # Write validated data to Delta table
    batch_df.write.format("delta").mode("append").saveAsTable(OUTPUT_TABLE)
    print(f"Batch {batch_id}: Written to Delta table '{OUTPUT_TABLE}'")

    return results


print(f"foreachBatch handler defined → writes to '{OUTPUT_TABLE}'")

In [ ]:
# -------------------------------------------------
# Run GX validation on sample customer data
#
# This exercises ALL 5 expectation rules including dob.
# In production, this same validate_with_gx function
# is called via writeStream.foreachBatch().
# -------------------------------------------------
from pyspark.sql import functions as F

# Drop table from previous runs (idempotent)
spark.sql(f"DROP TABLE IF EXISTS {OUTPUT_TABLE}")

# Generate realistic sample data (includes dob to exercise all expectations)
sample_data = spark.range(25).select(
    F.col("id").cast("string").alias("party_key"),
    F.current_timestamp().alias("source_updated_at"),
    F.lit("Alice").alias("name"),
    F.lit("1990-05-15").alias("dob"),
    F.lit("US").alias("country"),
    F.current_timestamp().alias("ingested_at"),
)

print(f"Generated {sample_data.count()} sample records")
sample_data.show(5, truncate=False)

# Run GX validation + write to Delta table
results = validate_with_gx(sample_data, batch_id=0)

# Show validation results
print("\n=== GX Validation Summary ===")
print(f"Overall success: {results['success']}")
print(f"Expectations evaluated: {len(results['results'])}")
for r in results["results"]:
    status = "PASS" if r["success"] else "FAIL"
    etype = r["expectation_config"]["expectation_type"]
    col = r["expectation_config"]["kwargs"].get("column", "")
    print(f"  [{status}] {etype} (column={col})")

# Verify data was written to Delta table
written = spark.table(OUTPUT_TABLE)
print(f"\nDelta table '{OUTPUT_TABLE}': {written.count()} rows written")
written.show(5, truncate=False)

In [ ]:
# -------------------------------------------------
# FAILURE DEMO: What happens when validation fails
#
# This cell shows the fail-fast behavior:
# - Bad data triggers immediate RuntimeError
# - No bad data is written to the Delta table
# - In Airflow, the task fails → notify_failure triggers
# -------------------------------------------------

# Create bad data: invalid country + duplicate party_keys + null name
bad_data = spark.createDataFrame(
    [
        ("1", "2024-01-01T10:00:00", "Alice", "1990-05-15", "US"),
        ("1", "2024-01-01T11:00:00", "Bob", "1990-06-20", "US"),  # duplicate party_key
        (
            "3",
            "2024-01-01T12:00:00",
            None,
            "1985-03-10",
            "INVALID",
        ),  # null name + bad country
    ],
    ["party_key", "source_updated_at", "name", "dob", "country"],
)

print("Bad data sample (has duplicates, invalid country, null name):")
bad_data.show(truncate=False)

# This WILL raise RuntimeError — that's the expected behavior
try:
    validate_with_gx(bad_data, batch_id=999)
    print("ERROR: Should not reach here!")
except RuntimeError as e:
    print(f"\n=== EXPECTED FAILURE ===")
    print(f"{e}")
    print(f"\nFail-fast behavior confirmed:")
    print(f"  - RuntimeError raised immediately")
    print(f"  - No bad data written to Delta table")
    print(f"  - In Airflow: task fails → notify_failure triggers")
    print(f"  - In streaming: query stops, batch can be retried")